In [1]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [2]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher

ModuleNotFoundError: No module named 'src.ghcn_daily.data_fetching'

In [3]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
cpu_config = config.get("settings.json", "cpu_config")


In [4]:
print(cpu_config)

{'cpu_usage_limit': 85, 'max_concurrent_workers': 12, 'max_concurrent_processes': 6, 'cpu_check_interval': 2, 'chunk_size': 100, 'dynamic_worker_adjustment': True}


In [5]:
data_fetcher = DataFetcher(async_fetch=True, max_workers=5, chunk_size=100, cpu_usage_limit=75)

In [6]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [7]:
s_state_list=stations[stations['STATE']=='NM']['ID'].tolist()
m_state_list = stations[stations['STATE'].isin(['NM', 'TX'])]['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024)]['ID'].unique().tolist()

In [8]:
len(s_state_list) , len(s_live_list),len(m_state_list)

(2295, 767, 8767)

In [10]:
df= await data_fetcher.save_to_dataframe(s_state_list)

Fetching Data: 100%|████████████████████████████████████████████████| 23/23 [01:41<00:00,  4.39s/it]


Data fetching and saving completed. Data saved to DataFrame.


In [11]:
df

,ID,YEAR,Month,ELEMENT,VALUE1,MFLAG1,QFLAG1,SFLAG1,VALUE2,MFLAG2,...,QFLAG29,SFLAG29,VALUE30,MFLAG30,QFLAG30,SFLAG30,VALUE31,MFLAG31,QFLAG31,SFLAG31
0,US1NMBR0002,2005,3,PRCP,-9999,,,,-9999,,...,,N,3,,,N,0,T,,N
1,US1NMBR0002,2005,3,SNOW,-9999,,,,-9999,,...,,N,0,,,N,0,,,N
2,US1NMBR0002,2005,3,SNWD,-9999,,,,-9999,,...,,N,0,,,N,0,,,N
3,US1NMBR0002,2005,3,WESD,-9999,,,,-9999,,...,,N,0,,,N,0,,,N
4,US1NMBR0002,2005,3,WESF,-9999,,,,-9999,,...,,N,0,,,N,0,,,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1883967,USW00093097,2025,3,WDF2,150,,,W,170,,...,,,-9999,,,,-9999,,,
1883968,USW00093097,2025,3,WDF5,160,,,W,170,,...,,,-9999,,,,-9999,,,
1883969,USW00093097,2025,3,WSF2,139,,,W,134,,...,,,-9999,,,,-9999,,,
1883970,USW00093097,2025,3,WSF5,206,,,W,206,,...,,,-9999,,,,-9999,,,
